In [1]:
import os, pickle

from physics.simulation import mcfm, msq
from physics.hzz import zpair, zz4l
from datasets import balanced
from models import carl

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import hist

import torch
from torch.utils.data import Dataset
import lightning

In [2]:
torch.set_float32_matmul_precision('highest')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
})


In [ ]:
OUTPUT_DIR = 'run/carl/'
SCALER_FILE_X = 'scaler.pkl'
SCALER_FILE_Y = None
CHECKPOINT_DIR = 'checkpoints'
SAMPLE_DIR = 'data/'

TRAIN_EPOCH = '174'
VAL_LOSS = 0.68
LOG_VERSION = 0

CHECKPOINT = f'checkpoint-carl-epoch={TRAIN_EPOCH}-val_loss={VAL_LOSS}.ckpt'
LIGHTNING_DIR = f'lightning_logs/version_{LOG_VERSION}'

SAMPLE_SIZE = 1200000

FEATURES=['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
          'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
          'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
          'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

BATCH_SIZE = 128
SEED = 42

In [4]:
events_numerator = mcfm.from_csv(cross_section=14.482054, file_path=os.path.join(SAMPLE_DIR, 'qqZZ2e2m/events.csv'))
events_denominator = mcfm.from_csv(cross_section=1.5569109, file_path=os.path.join(SAMPLE_DIR, 'ggZZ2e2m_sbi/events.csv'))

z_cand = zpair.ZPairCandidate(algorithm='leastsquare')
z_masses = zpair.ZPairMassWindow(z1=(70,115), z2=(70,115))
angles = zz4l.AngularVariables()
four_lepton_vars = zz4l.FourLeptonSystem()
lepton_momenta = zz4l.LeptonMomenta()

events_numerator = events_numerator.calculate(z_cand).filter(z_masses).calculate(angles).calculate(four_lepton_vars).calculate(lepton_momenta)
events_denominator = events_denominator.calculate(z_cand).filter(z_masses).calculate(angles).calculate(four_lepton_vars).calculate(lepton_momenta)

train_size, val_size, test_size = 1.0, 0.1, 0.1
events_numerator_train, events_numerator_val, events_numerator_test = events_numerator.unweight(SAMPLE_SIZE,random_state=SEED).split(train_size=train_size, val_size=val_size, test_size=test_size)
events_denominator_train, events_denominator_val, events_denominator_test = events_denominator.unweight(SAMPLE_SIZE,random_state=SEED).split(train_size=train_size, val_size=val_size, test_size=test_size)

training_data = balanced.BalancedDataset(events_numerator_train, events_denominator_train, FEATURES)
validation_data = balanced.BalancedDataset(events_numerator_val, events_denominator_val, FEATURES)
testing_data = balanced.BalancedDataset(events_numerator_test, events_denominator_test, FEATURES)

with open(os.path.join(OUTPUT_DIR, SCALER_FILE_X), 'rb') as f:
    scaler_X = pickle.load(f)

training_data.X = scaler_X.transform(training_data.X)
validation_data.X = scaler_X.transform(validation_data.X)
testing_data.X = scaler_X.transform(testing_data.X)

In [5]:
loaded_model = carl.CARL.load_from_checkpoint(os.path.join(OUTPUT_DIR, CHECKPOINT_DIR, CHECKPOINT))

dl_train = torch.utils.data.DataLoader(training_data, batch_size=BATCH_SIZE)
dl_val = torch.utils.data.DataLoader(validation_data, batch_size=BATCH_SIZE)
dl_test = torch.utils.data.DataLoader(testing_data, batch_size=BATCH_SIZE)

trainer = lightning.Trainer(accelerator='gpu', devices=1)

predictions_train = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_train]), axis=0).detach().numpy()
predictions_val = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_val]), axis=0).detach().numpy()
predictions_test = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_test]), axis=0).detach().numpy()

ratios_train_pred = predictions_train/(1-predictions_train)
ratios_val_pred = predictions_val/(1-predictions_val)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA H200') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` whic

Predicting: |                                                                                                 …

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [6]:
targets_val = validation_data.s
targets_test = testing_data.s

In [7]:
bins = 60
bounds = [0,1]
step_size = (bounds[1]-bounds[0])/bins

p = predictions_test
t = targets_test

pred_binned = [p[(p > bounds[0]+step_size*i) & (p <= bounds[0]+step_size*(i+1))] for i in range(bins)]
targets_binned = [t[(p > bounds[0]+step_size*i) & (p <= bounds[0]+step_size*(i+1))] for i in range(bins)]

sig_per_bin = np.array([np.sum(targets_binned[i]==1.0) for i in range(bins)])
bkg_per_bin = np.array([np.sum(targets_binned[i]==0.0) for i in range(bins)])

s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)

sig_err = np.array([np.std(targets_binned[i]==1.0) for i in range(bins)])
bkg_err = np.array([np.std(targets_binned[i]==0.0) for i in range(bins)])
s_err = np.sqrt((sig_err * bkg_per_bin/(sig_per_bin + bkg_per_bin)**2)**2 + (-bkg_err*sig_per_bin/(sig_per_bin + bkg_per_bin)**2)**2)

s_centers = [bounds[0]+(i+1/2)*step_size for i in range(bins)]

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(4,5), sharex=True)

ax1.errorbar(s_centers, s_centers, color='black', linestyle='--', label='$\mathrm{MC}$ $\mathrm{estimate}$')
ax1.errorbar(s_centers, s_true, yerr=s_err, color='royalblue', marker='o', markersize=4, linestyle='none', label='$\mathrm{NSBI}$ $\mathrm{estimate}$')

ax1.set_ylim(*bounds)
ax1.set_ylabel('$s(x) = \\frac{p_{q\\bar{q}}(x)}{ p_{q\\bar{q}}(x) + p_{gg}(x) }$')

ax1.legend(frameon=False)

ax2.errorbar(s_centers, np.array(s_centers)-np.array(s_centers), color='black', linestyle='--')
ax2.errorbar(s_centers, np.array(s_true)-np.array(s_centers), yerr=0., color='royalblue', marker='o', markersize=4, linestyle='none')

ax2.set_xlim(*bounds)
ax2.set_xlabel('$\\hat{s}(x)$')
ax2.set_ylabel('$\\mathrm{Residual}$')
ax2.set_ylim(-0.5, 0.5)

plt.tight_layout()
plt.savefig('carl_calibration.pdf', bbox_inches='tight')
plt.close()

/tmp/ipykernel_1869376/2792673948.py:14: RuntimeWarning: invalid value encountered in divide
  s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
